# LeetCode 2192 — All Ancestors of a Node in a Directed Acyclic Graph bằng Python

Link đề: https://leetcode.com/problems/all-ancestors-of-a-node-in-a-directed-acyclic-graph/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Graph, Topological Sort

Skeleton LeetCode:

```python
class Solution:
    def getAncestors(self, n: int, edges: List[List[int]]) -> List[List[int]]:
        ...
```

Với mỗi đỉnh của DAG, liệt kê mọi **tổ tiên** (đỉnh đi tới được nó) theo thứ tự tăng. Đây là bài **bao đóng bắc cầu** (transitive closure). Notebook đi từ **BFS ngược từ mỗi đỉnh** → **DFS xuôi từ mỗi đỉnh nguồn, phát tán chính nó** → **thứ tự tô-pô + hợp tập** → **thứ tự tô-pô + bitset (số nguyên Python)** → **Warshall trên bitset**.

## 1. Mục tiêu học

1. Hiểu "tổ tiên của `v`" = mọi `u ≠ v` có đường đi `u ⇝ v` ⇒ tính bằng duyệt **đồ thị ngược** từ `v`.
2. Dùng **thứ tự tô-pô** (Kahn) để tính `anc[v] = ∪_{u → v} (anc[u] ∪ {u})` khi mọi tiền bối đã xong.
3. Dùng **bitset** (số nguyên Python) để hợp tập nhanh: `anc[v] |= anc[u] | (1 << u)`.
4. Tránh lỗi: trùng lặp, chưa sắp, xử lý theo **chỉ số** thay vì thứ tự tô-pô, DFS đệ quy quá sâu.
5. Kiểm thử đối chiếu nhiều cách trên DAG ngẫu nhiên (sinh bằng cách chỉ cho cạnh từ thứ hạng nhỏ sang lớn).

## 2. Đề bài & ví dụ

Cho DAG `n` đỉnh `0..n−1` và danh sách cạnh `edges[i] = [from, to]`. Trả về `answer` với `answer[i]` là danh sách **tổ tiên** của đỉnh `i`, **sắp tăng**. `u` là tổ tiên của `v` nếu từ `u` đi tới được `v` qua các cạnh.

**Ví dụ 1:** `n = 8`, cạnh `[[0,3],[0,4],[1,3],[2,4],[2,7],[3,5],[3,6],[3,7],[4,6]]` → `[[],[],[],[0,1],[0,2],[0,1,3],[0,1,2,3,4],[0,1,2,3]]`.

**Ví dụ 2:** `n = 5`, cạnh đầy đủ `i → j` với `i < j` → `[[],[0],[0,1],[0,1,2],[0,1,2,3]]`.

**Ràng buộc:** `1 ≤ n ≤ 1000`, `0 ≤ len(edges) ≤ min(2000, n(n−1)/2)`, không cạnh trùng, đồ thị không chu trình.

In [1]:
import random
import sys
import time
from collections import deque
from typing import List

examples = [
    (8, [[0, 3], [0, 4], [1, 3], [2, 4], [2, 7], [3, 5], [3, 6], [3, 7], [4, 6]],
     [[], [], [], [0, 1], [0, 2], [0, 1, 3], [0, 1, 2, 3, 4], [0, 1, 2, 3]]),
    (5, [[0, 1], [0, 2], [0, 3], [0, 4], [1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]],
     [[], [0], [0, 1], [0, 1, 2], [0, 1, 2, 3]]),
]

def adjacency(n, edges):
    out = [[] for _ in range(n)]; inc = [[] for _ in range(n)]
    for u, v in edges:
        out[u].append(v); inc[v].append(u)
    return out, inc

out1, inc1 = adjacency(*examples[0][:2])
print("cạnh ra:", out1)
print("cạnh vào:", inc1)

cạnh ra: [[3, 4], [3], [4, 7], [5, 6, 7], [6], [], [], []]
cạnh vào: [[], [], [], [0, 1], [0, 2], [3], [3, 4], [2, 3]]


## 3. Kiến thức nền

### 3.1 Đồ thị ngược

Đảo mọi cạnh `u → v` thành `v → u`. Tổ tiên của `v` trong đồ thị gốc = các đỉnh đi tới được **từ** `v` trong đồ thị ngược.

### 3.2 Thứ tự tô-pô (Kahn)

In [2]:
def topo_order(n, edges):
    out, _ = adjacency(n, edges)
    indeg = [0] * n
    for u, v in edges:
        indeg[v] += 1
    q = deque(i for i in range(n) if indeg[i] == 0)
    order = []
    while q:
        u = q.popleft(); order.append(u)
        for v in out[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return order

print("thứ tự tô-pô ví dụ 1:", topo_order(*examples[0][:2]))

thứ tự tô-pô ví dụ 1: [0, 1, 2, 3, 4, 5, 7, 6]


### 3.3 Bitset bằng số nguyên Python

Tập `S ⊆ {0..n−1}` ↔ số `x = Σ_{i∈S} 2^i`. Hợp: `x | y`; thêm `u`: `x | (1 << u)`; liệt kê phần tử tăng dần: duyệt bit thấp nhất `x & −x`.

In [3]:
def bits_ascending(x):
    out = []
    while x:
        low = x & -x
        out.append(low.bit_length() - 1)
        x ^= low
    return out

a = (1 << 0) | (1 << 3); b = (1 << 1) | (1 << 3)
print("a ∪ b =", bits_ascending(a | b), "| thêm 5:", bits_ascending(a | b | 1 << 5))

a ∪ b = [0, 1, 3] | thêm 5: [0, 1, 3, 5]


### 3.4 Sinh DAG ngẫu nhiên

Chọn một hoán vị ngẫu nhiên làm "thứ hạng"; chỉ thêm cạnh từ thứ hạng nhỏ sang lớn ⇒ chắc chắn không có chu trình.

In [4]:
def random_dag(n, m, rnd):
    perm = list(range(n)); rnd.shuffle(perm)
    rank = {v: i for i, v in enumerate(perm)}
    edges = set()
    max_edges = n * (n - 1) // 2
    m = min(m, max_edges)
    while len(edges) < m:
        u, v = rnd.sample(range(n), 2)
        if rank[u] > rank[v]:
            u, v = v, u
        edges.add((u, v))
    return [list(e) for e in edges]

print(random_dag(5, 4, random.Random(1)))

[[0, 1], [4, 0], [3, 4], [3, 0]]


## 4. Ý tưởng nền tảng / trực giác

Với ví dụ 1, nhìn đỉnh 6: cạnh vào từ `3` và `4`. Tổ tiên của 6 = `{3, 4}` ∪ tổ tiên của 3 (`{0, 1}`) ∪ tổ tiên của 4 (`{0, 2}`) = `{0, 1, 2, 3, 4}` ✓.

Công thức đệ quy: `anc[v] = ∪_{u → v} ({u} ∪ anc[u])`. Để dùng được, `anc[u]` phải **xong trước** `anc[v]` với mọi cạnh `u → v` ⇒ xử lý theo **thứ tự tô-pô**.

Cách nghĩ ngược: mỗi đỉnh `u` "phát tán" chính nó tới **mọi** đỉnh nó đi tới được ⇒ DFS/BFS xuôi từ `u`, thêm `u` vào `anc` của mọi đỉnh thăm được. Nếu duyệt `u = 0, 1, 2, …` thì danh sách `anc[v]` **tự động tăng dần** — không cần sắp!

## 5. Cách 1 — Brute Force / baseline: BFS trên đồ thị ngược từ mỗi đỉnh

In [5]:
def ancestors_bfs_reverse(n, edges):
    _, inc = adjacency(n, edges)
    ans = []
    for v in range(n):
        seen = set(); q = deque(inc[v])
        seen.update(inc[v])
        while q:
            u = q.popleft()
            for w in inc[u]:
                if w not in seen:
                    seen.add(w); q.append(w)
        ans.append(sorted(seen))
    return ans

for n_, e_, exp in examples:
    print(ancestors_bfs_reverse(n_, e_) == exp, ancestors_bfs_reverse(n_, e_))

True [[], [], [], [0, 1], [0, 2], [0, 1, 3], [0, 1, 2, 3, 4], [0, 1, 2, 3]]
True [[], [0], [0, 1], [0, 1, 2], [0, 1, 2, 3]]


**Độ phức tạp:** `O(n · (n + m) + Σ|anc| log)` — với `n = 1000`, `m = 2000` khoảng `3·10⁶` bước: chạy được. Đơn giản, đúng hiển nhiên ⇒ chuẩn đối chiếu.

**Nút thắt:** mỗi đỉnh duyệt lại toàn bộ phần đồ thị phía trên nó — các đỉnh chung tổ tiên bị duyệt lặp. Thứ tự tô-pô cho phép **tái dùng** kết quả.

## 6. Ý tưởng thuật toán / bước đột phá

**Bất biến tô-pô:** khi xử lý `v` theo thứ tự tô-pô, mọi `u` có cạnh `u → v` đã có `anc[u]` hoàn chỉnh.

**Quy tắc:** với mỗi cạnh `u → v` (duyệt khi lấy `u` ra khỏi hàng đợi Kahn): `anc[v] |= anc[u] | {u}`.

**Bitset:** tập là số nguyên ⇒ hợp tập `O(n / 64)` phép máy (Python xử lý số lớn theo khối 30 bit). Cuối cùng liệt kê bit ⇒ đã **tăng dần**.

**Warshall:** `reach[i]` là bitset các đỉnh `i` đi tới được; với mỗi `k`, mọi `i` đi tới được `k` thì `reach[i] |= reach[k]`. `O(n²)` phép bitset — vẫn chạy với `n = 1000`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mỗi đỉnh "phát tán" chính nó (DFS xuôi lặp, không cần sắp)

In [6]:
def ancestors_spread(n, edges):
    out, _ = adjacency(n, edges)
    ans = [[] for _ in range(n)]
    for u in range(n):                         # u tăng dần -> mỗi ans[v] tự tăng dần
        seen = [False] * n
        stack = list(out[u])
        for v in stack:
            seen[v] = True
        while stack:
            v = stack.pop()
            ans[v].append(u)
            for w in out[v]:
                if not seen[w]:
                    seen[w] = True; stack.append(w)
    return ans

for n_, e_, exp in examples:
    print(ancestors_spread(n_, e_) == exp)

True
True


**Độ phức tạp:** `O(n · (n + m))` — như cách 1, nhưng không cần sắp (thứ tự `u` tăng dần) và DFS lặp nên không lo giới hạn đệ quy.

### 7.2 Cách 3 — Thứ tự tô-pô + hợp tập `set`

In [7]:
def ancestors_topo_set(n, edges):
    out, _ = adjacency(n, edges)
    indeg = [0] * n
    for u, v in edges:
        indeg[v] += 1
    anc = [set() for _ in range(n)]
    q = deque(i for i in range(n) if indeg[i] == 0)
    while q:
        u = q.popleft()
        for v in out[u]:
            anc[v] |= anc[u]
            anc[v].add(u)
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return [sorted(s) for s in anc]

for n_, e_, exp in examples:
    print(ancestors_topo_set(n_, e_) == exp)

True
True


**Độ phức tạp:** `O(m · n)` cho các phép hợp (mỗi cạnh hợp một tập cỡ `≤ n`) + `O(Σ|anc| log)` sắp. Bộ nhớ `O(Σ|anc|) ≤ O(n²)`.

### 7.3 Cách 4 — Thứ tự tô-pô + bitset (số nguyên)

In [8]:
def ancestors_topo_bits(n, edges):
    out, _ = adjacency(n, edges)
    indeg = [0] * n
    for u, v in edges:
        indeg[v] += 1
    anc = [0] * n
    q = deque(i for i in range(n) if indeg[i] == 0)
    while q:
        u = q.popleft()
        mask = anc[u] | (1 << u)
        for v in out[u]:
            anc[v] |= mask
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return [bits_ascending(x) for x in anc]

for n_, e_, exp in examples:
    print(ancestors_topo_bits(n_, e_) == exp)

True
True


**Độ phức tạp:** `O(m · n/w)` cho hợp (`w` ≈ 30–64 bit mỗi "chữ số") + `O(Σ|anc|)` liệt kê. Thường nhanh nhất trong Python.

### 7.4 Cách 5 — Bao đóng bắc cầu Warshall trên bitset

In [9]:
def ancestors_warshall(n, edges):
    reach = [0] * n                            # reach[i]: bitset các đỉnh i đi tới được
    for u, v in edges:
        reach[u] |= 1 << v
    for k in range(n):
        bk = 1 << k; rk = reach[k]
        for i in range(n):
            if reach[i] & bk:
                reach[i] |= rk
    ans = [[] for _ in range(n)]
    for u in range(n):                         # u tăng dần -> ans tự tăng
        for v in bits_ascending(reach[u]):
            ans[v].append(u)
    return ans

for n_, e_, exp in examples:
    print(ancestors_warshall(n_, e_) == exp)

True
True


**Vì sao đúng khi cập nhật tại chỗ?** Warshall: sau vòng `k`, `reach[i]` chứa mọi đỉnh tới được qua các đỉnh trung gian thuộc `{0..k}`. `reach[k]` không bị đổi trong vòng `k` (vì `reach[k] & bk` bằng 0 trong DAG — không có chu trình qua `k`), nên đọc `rk` trước vòng `i` là an toàn. **Độ phức tạp:** `O(n²)` phép bitset `O(n/w)` ⇒ `O(n³/w)`; với `n = 1000` khoảng `10⁶` phép — chấp nhận được, không phụ thuộc số cạnh.

## 8. TRACE từng bước

### 8.1 Trace cách 4 (tô-pô + bitset) trên ví dụ 1

In [10]:
def trace_topo_bits(n, edges):
    out, _ = adjacency(n, edges); indeg = [0] * n
    for u, v in edges: indeg[v] += 1
    anc = [0] * n; q = deque(i for i in range(n) if indeg[i] == 0)
    print("  nguồn ban đầu:", list(q))
    while q:
        u = q.popleft(); mask = anc[u] | (1 << u)
        for v in out[u]:
            anc[v] |= mask; indeg[v] -= 1
            note = " -> vào hàng đợi" if indeg[v] == 0 else ""
            print(f"    lấy {u}: cạnh {u}->{v}, anc[{v}] = {bits_ascending(anc[v])}{note}")
            if indeg[v] == 0: q.append(v)
    return [bits_ascending(x) for x in anc]

r = trace_topo_bits(*examples[0][:2])
print("  =>", r)

  nguồn ban đầu: [0, 1, 2]
    lấy 0: cạnh 0->3, anc[3] = [0]
    lấy 0: cạnh 0->4, anc[4] = [0]
    lấy 1: cạnh 1->3, anc[3] = [0, 1] -> vào hàng đợi
    lấy 2: cạnh 2->4, anc[4] = [0, 2] -> vào hàng đợi
    lấy 2: cạnh 2->7, anc[7] = [2]
    lấy 3: cạnh 3->5, anc[5] = [0, 1, 3] -> vào hàng đợi
    lấy 3: cạnh 3->6, anc[6] = [0, 1, 3]
    lấy 3: cạnh 3->7, anc[7] = [0, 1, 2, 3] -> vào hàng đợi
    lấy 4: cạnh 4->6, anc[6] = [0, 1, 2, 3, 4] -> vào hàng đợi
  => [[], [], [], [0, 1], [0, 2], [0, 1, 3], [0, 1, 2, 3, 4], [0, 1, 2, 3]]


### 8.2 Trace cách 2 (phát tán) trên DAG tự dựng 6 đỉnh (đã đối chiếu BFS ngược — đối chiếu lại đề gốc)

In [11]:
def trace_spread(n, edges):
    out, _ = adjacency(n, edges); ans = [[] for _ in range(n)]
    for u in range(n):
        seen = [False] * n; stack = list(out[u]); reached = []
        for v in stack: seen[v] = True
        while stack:
            v = stack.pop(); ans[v].append(u); reached.append(v)
            for w in out[v]:
                if not seen[w]: seen[w] = True; stack.append(w)
        print(f"    đỉnh {u} phát tán tới {sorted(reached)}")
    return ans

demo_n, demo_e = 6, [[5, 0], [5, 2], [0, 1], [2, 1], [1, 3], [4, 3]]
r = trace_spread(demo_n, demo_e)
print("  =>", r, "| BFS ngược:", ancestors_bfs_reverse(demo_n, demo_e))

    đỉnh 0 phát tán tới [1, 3]
    đỉnh 1 phát tán tới [3]
    đỉnh 2 phát tán tới [1, 3]
    đỉnh 3 phát tán tới []
    đỉnh 4 phát tán tới [3]
    đỉnh 5 phát tán tới [0, 1, 2, 3]
  => [[5], [0, 2, 5], [5], [0, 1, 2, 4, 5], [], []] | BFS ngược: [[5], [0, 2, 5], [5], [0, 1, 2, 4, 5], [], []]


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def getAncestors(self, n: int, edges: List[List[int]]) -> List[List[int]]:
        out = [[] for _ in range(n)]
        indeg = [0] * n
        for u, v in edges:
            out[u].append(v)
            indeg[v] += 1
        anc = [0] * n                               # bitset tổ tiên
        q = deque(i for i in range(n) if indeg[i] == 0)
        while q:                                    # Kahn: tiền bối luôn xong trước
            u = q.popleft()
            mask = anc[u] | (1 << u)
            for v in out[u]:
                anc[v] |= mask
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        res = []
        for x in anc:
            lst = []
            while x:
                low = x & -x
                lst.append(low.bit_length() - 1)    # bit thấp trước -> tăng dần
                x ^= low
            res.append(lst)
        return res

for n_, e_, exp in examples:
    assert Solution().getAncestors(n_, e_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Cách 4 / code nộp.** Gọi `A(v)` là tập tổ tiên thật. Ta có `A(v) = ∪_{u → v} ({u} ∪ A(u))` (mọi đường tới `v` đi qua một cạnh cuối `u → v`).

Kahn lấy `u` ra khỏi hàng đợi chỉ khi mọi cạnh vào `u` đã được xử lý (bậc vào về 0). Quy nạp theo thứ tự lấy ra: khi `u` được lấy, `anc[u] = A(u)` (mọi tiền bối `p` của `u` đã được lấy trước và đã đóng góp `{p} ∪ A(p)`). Khi đó với mỗi cạnh `u → v`, ta cộng `{u} ∪ A(u)` vào `anc[v]`. Khi `v` được lấy ra, mọi cạnh vào đã đóng góp ⇒ `anc[v] = A(v)`. Vì đồ thị là DAG, mọi đỉnh đều được lấy ra.

Liệt kê bit từ thấp lên ⇒ danh sách tăng dần; bitset không có trùng lặp.

**Cách 2.** Mỗi `u` thêm mình vào đúng các đỉnh nó tới được (DFS xuôi, `seen` tránh thăm lặp) ⇒ `ans[v] = {u : u ⇝ v}`; `u` duyệt tăng nên danh sách tăng.

## 11. Phân tích Time / Space Complexity

Gọi `n` đỉnh, `m` cạnh, `w` = độ rộng "chữ số" số nguyên lớn (≈ 30 bit), `S = Σ|anc|` (≤ `n²/2`).

| Cách | Time | Space |
|---|---|---|
| 1. BFS ngược từ mỗi đỉnh | `O(n(n + m) + S log n)` | `O(n + m)` mỗi lần |
| 2. Phát tán (DFS xuôi) | `O(n(n + m))` | `O(n + m + S)` |
| 3. Tô-pô + `set` | `O(m n + S log n)` | `O(S)` |
| 4. Tô-pô + bitset | `O(m n / w + S)` | `O(n² / w)` |
| 5. Warshall bitset | `O(n³ / w + S)` | `O(n² / w)` |

In [13]:
rnd = random.Random(2192)
cases = {"thưa (n=1000, m=2000)": (1000, random_dag(1000, 2000, rnd)),
         "dây xích (n=1000, m=999)": (1000, [[i, i + 1] for i in range(999)]),
         "dày (n=64, m=2000)": (64, random_dag(64, 2000, rnd))}
for label_, (n_, e_) in cases.items():
    row = []
    for name, fn in [("bfs_rev", ancestors_bfs_reverse), ("spread", ancestors_spread), ("topo_set", ancestors_topo_set),
                     ("topo_bits", ancestors_topo_bits), ("warshall", ancestors_warshall), ("Solution", Solution().getAncestors)]:
        t0 = time.perf_counter(); r = fn(n_, e_); row.append(f"{name}={(time.perf_counter() - t0) * 1000:.0f}ms")
    print(f"{label_:<26} (Σ|anc| = {sum(map(len, r))}): " + ", ".join(row))

thưa (n=1000, m=2000)      (Σ|anc| = 10880): bfs_rev=3ms, spread=2ms, topo_set=2ms, topo_bits=3ms, warshall=52ms, Solution=2ms


dây xích (n=1000, m=999)   (Σ|anc| = 499500): bfs_rev=39ms, spread=46ms, topo_set=19ms, topo_bits=62ms, warshall=130ms, Solution=60ms
dày (n=64, m=2000)         (Σ|anc| = 2015): bfs_rev=3ms, spread=1ms, topo_set=1ms, topo_bits=0ms, warshall=1ms, Solution=0ms


## 12. Bộ test

In [14]:
ALL = {
    "spread": ancestors_spread,
    "topo_set": ancestors_topo_set,
    "topo_bits": ancestors_topo_bits,
    "warshall": ancestors_warshall,
    "Solution": lambda n, e: Solution().getAncestors(n, e),
}

def check(n, edges, exp=None):
    ref = ancestors_bfs_reverse(n, edges)
    for name, fn in ALL.items():
        got = fn(n, [list(e) for e in edges])
        assert got == ref, (name, n, edges, got, ref)
    if exp is not None:
        assert ref == exp
    return ref

for n_, e_, exp in examples:
    check(n_, e_, exp)
print("Ví dụ đề: OK")

check(1, [], [[]])                                     # một đỉnh
check(3, [], [[], [], []])                             # không cạnh
check(4, [[3, 2], [2, 1], [1, 0]], [[1, 2, 3], [2, 3], [3], []])    # dây xích ngược chỉ số
check(4, [[0, 3], [1, 3], [2, 3]], [[], [], [], [0, 1, 2]])
check(5, [[0, 1], [1, 2], [0, 2], [3, 4]])
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(500):
    n_ = rnd.randint(1, 12)
    check(n_, random_dag(n_, rnd.randint(0, n_ * (n_ - 1) // 2), rnd))
print("500 DAG ngẫu nhiên nhỏ: OK")

for _ in range(4):
    n_ = rnd.randint(200, 1000)
    check(n_, random_dag(n_, rnd.randint(0, 2000), rnd))
print("4 DAG lớn: OK")
check(1000, [[i, i + 1] for i in range(999)])
print("Dây xích 1000 đỉnh: OK")

Ví dụ đề: OK
Edge case: OK
500 DAG ngẫu nhiên nhỏ: OK
4 DAG lớn: OK


Dây xích 1000 đỉnh: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ lấy **cha trực tiếp** (quên tính bắc cầu)

In [15]:
def bug_direct_parents(n, edges):
    _, inc = adjacency(n, edges)
    return [sorted(p) for p in inc]

n_, e_, exp = examples[0]
print("đỉnh 6 -> sai:", bug_direct_parents(n_, e_)[6], "| đúng:", Solution().getAncestors(n_, e_)[6])

đỉnh 6 -> sai: [3, 4] | đúng: [0, 1, 2, 3, 4]


**Sửa:** tổ tiên gồm mọi đỉnh tới được, không chỉ cạnh vào.

### Lỗi 2 — Xử lý theo **chỉ số** `0..n−1` thay vì thứ tự tô-pô

In [16]:
def bug_index_order(n, edges):
    out, _ = adjacency(n, edges)
    anc = [0] * n
    for u in range(n):                              # SAI: anc[u] có thể chưa hoàn chỉnh
        for v in out[u]:
            anc[v] |= anc[u] | (1 << u)
    return [bits_ascending(x) for x in anc]

n_, e_ = 4, [[3, 2], [2, 1], [1, 0]]
print("sai :", bug_index_order(n_, e_), "| đúng:", Solution().getAncestors(n_, e_))

sai : [[1], [2], [3], []] | đúng: [[1, 2, 3], [2, 3], [3], []]


**Vì sao sai:** đỉnh 1 được xử lý trước đỉnh 2, 3 — lúc đó `anc[1]` còn rỗng nên đỉnh 0 chỉ nhận `{1}`. Ví dụ của đề tình cờ có thứ tự chỉ số trùng thứ tự tô-pô nên **không** lộ lỗi. **Sửa:** Kahn hoặc DFS hậu thứ tự.

### Lỗi 3 — Gộp danh sách mà không khử trùng

In [17]:
def bug_duplicates(n, edges):
    out, _ = adjacency(n, edges); indeg = [0] * n
    for u, v in edges: indeg[v] += 1
    anc = [[] for _ in range(n)]; q = deque(i for i in range(n) if indeg[i] == 0)
    while q:
        u = q.popleft()
        for v in out[u]:
            anc[v] += anc[u] + [u]                  # SAI: tổ tiên chung bị thêm nhiều lần
            indeg[v] -= 1
            if indeg[v] == 0: q.append(v)
    return [sorted(a) for a in anc]

n_, e_, exp = examples[0]
print("đỉnh 6 -> sai:", bug_duplicates(n_, e_)[6], "| đúng:", Solution().getAncestors(n_, e_)[6])

đỉnh 6 -> sai: [0, 0, 1, 2, 3, 4] | đúng: [0, 1, 2, 3, 4]


**Vì sao sai:** đỉnh 0 tới 6 qua cả 3 và 4 ⇒ xuất hiện hai lần. Với DAG dày, danh sách còn phình to theo **số đường đi** (có thể mũ). **Sửa:** `set` hoặc bitset.

### Lỗi 4 — Quên sắp kết quả (khi dùng `set`)

In [18]:
def bug_unsorted(n, edges):
    out, _ = adjacency(n, edges); indeg = [0] * n
    for u, v in edges: indeg[v] += 1
    anc = [set() for _ in range(n)]; q = deque(i for i in range(n) if indeg[i] == 0)
    order_added = [[] for _ in range(n)]
    while q:
        u = q.popleft()
        for v in out[u]:
            for x in list(anc[u]) + [u]:
                if x not in anc[v]:
                    anc[v].add(x); order_added[v].append(x)      # SAI: trả theo thứ tự thêm vào
            indeg[v] -= 1
            if indeg[v] == 0: q.append(v)
    return order_added

n_, e_ = 4, [[3, 2], [2, 1], [1, 0]]
print("sai :", bug_unsorted(n_, e_), "| đúng:", Solution().getAncestors(n_, e_))

sai : [[2, 3, 1], [3, 2], [3], []] | đúng: [[1, 2, 3], [2, 3], [3], []]


**Sửa:** `sorted(...)` — hoặc dùng bitset / phát tán theo `u` tăng dần để tự có thứ tự.

### Lỗi 5 — DFS **đệ quy** trên dây xích dài

In [19]:
def recursion_needed_chain(n):
    return n - 1                                    # độ sâu đệ quy khi DFS xuôi trên dây xích 0 -> 1 -> ... -> n-1

print("dây xích 1000 đỉnh cần độ sâu", recursion_needed_chain(1000), "| giới hạn mặc định", sys.getrecursionlimit(),
      "->", "an toàn" if recursion_needed_chain(1000) < sys.getrecursionlimit() - 50 else "nguy hiểm (sát/vượt giới hạn)")

dây xích 1000 đỉnh cần độ sâu 999 | giới hạn mặc định 1000 -> nguy hiểm (sát/vượt giới hạn)


**Vì sao là lỗi:** với `n = 1000`, DFS đệ quy sâu `999` cộng các khung gọi của chính notebook/LeetCode ⇒ **sát** giới hạn 1000 và dễ `RecursionError`. Ta không chạy bản đệ quy để giữ notebook sạch. **Sửa:** DFS dùng stack (cách 2) hoặc Kahn.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mọi tổ tiên / mọi đỉnh tới được" trên DAG ⇒ **thứ tự tô-pô + hợp tập**, hoặc **duyệt từ mỗi đỉnh**.
- Tập con của `{0..n−1}` với `n ≤ vài nghìn` ⇒ **bitset bằng int** của Python (hợp/giao rất nhanh).
- Muốn danh sách tăng dần mà không sắp ⇒ để đỉnh nguồn duyệt theo chỉ số tăng (cách 2) hoặc liệt kê bit thấp trước.
- Test có đồ thị mà **thứ tự chỉ số khác thứ tự tô-pô** (cạnh từ chỉ số lớn sang nhỏ).
- Python: tránh DFS đệ quy khi `n` ~ 1000.

## 15. Biến thể của bài

1. **Con cháu** của mỗi đỉnh — đảo chiều cạnh.
2. **LeetCode 1462 — Course Schedule IV:** truy vấn "`u` có phải điều kiện tiên quyết của `v`?" — bao đóng bắc cầu (Warshall bitset).
3. **LeetCode 851 — Loud and Rich:** trong các tổ tiên (người giàu hơn), tìm người im lặng nhất — DP trên DAG.
4. **Đếm số tổ tiên** (không cần liệt kê) — `popcount` của bitset.
5. **Đồ thị có chu trình**: tổ tiên = mọi đỉnh tới được (có thể gồm chính nó) — dùng BFS từ mỗi đỉnh hoặc nén thành phần liên thông mạnh (SCC) rồi làm trên DAG nén.

In [20]:
def course_schedule_iv(numCourses, prerequisites, queries):
    reach = [0] * numCourses
    for u, v in prerequisites:
        reach[u] |= 1 << v
    for k in range(numCourses):
        bk = 1 << k; rk = reach[k]
        for i in range(numCourses):
            if reach[i] & bk:
                reach[i] |= rk
    return [bool(reach[u] >> v & 1) for u, v in queries]

def ancestor_counts(n, edges):
    return [len(a) for a in Solution().getAncestors(n, edges)]

print("1462:", course_schedule_iv(3, [[1, 2], [1, 0], [2, 0]], [[1, 0], [1, 2], [0, 1]]))
print("số tổ tiên ví dụ 1:", ancestor_counts(*examples[0][:2]))

1462: [True, True, False]
số tổ tiên ví dụ 1: [0, 0, 0, 2, 2, 3, 5, 4]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS ngược | duyệt từ mỗi đỉnh | `O(n(n+m))` | `O(n+m)` | chuẩn đối chiếu |
| 2. Phát tán | DFS xuôi, `u` tăng | `O(n(n+m))` | `O(S)` | không cần sắp |
| 3. Tô-pô + set | hợp tập | `O(mn)` | `O(S)` | dễ hiểu |
| 4. Tô-pô + bitset | hợp bằng `|` | `O(mn/w)` | `O(n²/w)` | **nộp bài** |
| 5. Warshall bitset | bao đóng | `O(n³/w)` | `O(n²/w)` | nhiều truy vấn, đồ thị dày |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().getAncestors(4, [[0,1],[1,2],[0,3]])`?

**Bài 2 (tính tay):** Với DAG tự dựng ở mục 8.2 (`5→0, 5→2, 0→1, 2→1, 1→3, 4→3`), viết một thứ tự tô-pô và `anc` của đỉnh 3.

**Bài 3 (điền TODO):** Hoàn thiện `descendants(n, edges)` — danh sách **con cháu** (tăng dần) của mỗi đỉnh.

**Bài 4 (tự viết lại):** Viết lại cách 4 nhưng tìm thứ tự tô-pô bằng **DFS hậu thứ tự lặp** (stack) thay vì Kahn.

**Bài 5 (mini challenge):** Với đồ thị **có thể có chu trình**, trả về mọi đỉnh `u ≠ v` tới được `v` cho mỗi `v`. Gợi ý: BFS ngược từ mỗi đỉnh (cẩn thận không thêm chính `v` dù có chu trình về `v`). Kiểm trên đồ thị ngẫu nhiên có chu trình bằng Warshall.

In [21]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def descendants(n, edges):
    # TODO: gọi Solution().getAncestors trên đồ thị đảo cạnh
    pass

# Bài 4 — TODO
def ancestors_dfs_topo(n, edges):
    # TODO: hậu thứ tự lặp -> đảo ngược = thứ tự tô-pô; rồi hợp bitset
    pass

# Bài 5 — TODO
def ancestors_general(n, edges):
    # TODO: BFS ngược từ mỗi v, bỏ v khỏi kết quả
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Đỉnh 0 không có tổ tiên.
- Mức 2: `1 ← 0`, `2 ← 1 ← 0`, `3 ← 0`.
- Mức 3: `[[], [0], [0, 1], [0]]`.

**Bài 2**
- Mức 1: Nguồn: `4, 5`.
- Mức 2: Một thứ tự: `4, 5, 0, 2, 1, 3`.
- Mức 3: `anc[3] = {0, 1, 2, 4, 5}`.

**Bài 3**
- Mức 1: Đảo mọi cạnh `[u, v] → [v, u]`.
- Mức 2: Tổ tiên trên đồ thị đảo = con cháu trên đồ thị gốc.
- Mức 3: Kiểm: `v ∈ desc[u] ⇔ u ∈ anc[v]`.

**Bài 4**
- Mức 1: Stack chứa `(đỉnh, chỉ số cạnh kế tiếp)`.
- Mức 2: Khi hết cạnh, thêm đỉnh vào `post`.
- Mức 3: `reversed(post)` là thứ tự tô-pô.

**Bài 5**
- Mức 1: Kahn không dùng được (có chu trình).
- Mức 2: BFS ngược từ `v`, `seen` khởi tạo rỗng, bỏ `v` ở cuối.
- Mức 3: Warshall bitset vẫn đúng với chu trình (đọc `reach[k]` mỗi lần trong vòng `i`).

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1
r1 = Solution().getAncestors(4, [[0, 1], [1, 2], [0, 3]])
print("Bài 1:", r1); assert r1 == [[], [0], [0, 1], [0]]

# Bài 2
print("Bài 2: thứ tự tô-pô (Kahn):", topo_order(demo_n, demo_e), "| anc[3] =", Solution().getAncestors(demo_n, demo_e)[3])
assert Solution().getAncestors(demo_n, demo_e)[3] == [0, 1, 2, 4, 5]

# Bài 3
def descendants_sol(n, edges):
    return Solution().getAncestors(n, [[v, u] for u, v in edges])

rnd = random.Random(3)
for _ in range(300):
    n_ = rnd.randint(1, 10); e_ = random_dag(n_, rnd.randint(0, 20), rnd)
    anc = Solution().getAncestors(n_, e_); desc = descendants_sol(n_, e_)
    assert all((v in desc[u]) == (u in anc[v]) for u in range(n_) for v in range(n_))
print("Bài 3 OK, ví dụ 1:", descendants_sol(*examples[0][:2]))

Bài 1: [[], [0], [0, 1], [0]]
Bài 2: thứ tự tô-pô (Kahn): [4, 5, 0, 2, 1, 3] | anc[3] = [0, 1, 2, 4, 5]
Bài 3 OK, ví dụ 1: [[3, 4, 5, 6, 7], [3, 5, 6, 7], [4, 6, 7], [5, 6, 7], [6], [], [], []]


In [23]:
# Bài 4
def ancestors_dfs_topo_sol(n, edges):
    out, _ = adjacency(n, edges)
    state = [0] * n; post = []
    for s in range(n):
        if state[s]: continue
        stack = [(s, 0)]; state[s] = 1
        while stack:
            v, i = stack.pop()
            if i < len(out[v]):
                stack.append((v, i + 1))
                w = out[v][i]
                if not state[w]:
                    state[w] = 1; stack.append((w, 0))
            else:
                post.append(v)
    anc = [0] * n
    for u in reversed(post):
        mask = anc[u] | (1 << u)
        for v in out[u]:
            anc[v] |= mask
    return [bits_ascending(x) for x in anc]

# Bài 5
def ancestors_general_sol(n, edges):
    _, inc = adjacency(n, edges); ans = []
    for v in range(n):
        seen = set(); q = deque([v])
        while q:
            u = q.popleft()
            for w in inc[u]:
                if w not in seen:
                    seen.add(w); q.append(w)
        seen.discard(v)
        ans.append(sorted(seen))
    return ans

def general_warshall(n, edges):
    reach = [0] * n
    for u, v in edges: reach[u] |= 1 << v
    for k in range(n):
        for i in range(n):
            if reach[i] >> k & 1:
                reach[i] |= reach[k]
    return [[u for u in range(n) if u != v and reach[u] >> v & 1] for v in range(n)]

rnd = random.Random(5)
for _ in range(300):
    n_ = rnd.randint(1, 10); e_ = random_dag(n_, rnd.randint(0, 20), rnd)
    assert ancestors_dfs_topo_sol(n_, e_) == Solution().getAncestors(n_, e_)
    g = [[rnd.randrange(n_), rnd.randrange(n_)] for _ in range(rnd.randint(0, 15))]
    g = [e for e in g if e[0] != e[1]]
    assert ancestors_general_sol(n_, g) == general_warshall(n_, g)
print("Bài 4, Bài 5 OK; đồ thị có chu trình 0->1->2->0, 2->3:", ancestors_general_sol(4, [[0, 1], [1, 2], [2, 0], [2, 3]]))

Bài 4, Bài 5 OK; đồ thị có chu trình 0->1->2->0, 2->3: [[1, 2], [0, 2], [0, 1], [0, 1, 2]]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template tô-pô + bitset:**

```python
anc = [0] * n; q = deque(i for i in range(n) if indeg[i] == 0)
while q:
    u = q.popleft(); mask = anc[u] | 1 << u
    for v in out[u]:
        anc[v] |= mask; indeg[v] -= 1
        if indeg[v] == 0: q.append(v)
# liệt kê bit thấp -> cao để có danh sách tăng dần
```

**Checklist:**
- [ ] Tính **bắc cầu**, không chỉ cha trực tiếp?
- [ ] Xử lý theo **thứ tự tô-pô** (không theo chỉ số)?
- [ ] Không trùng lặp (set/bitset)?
- [ ] Danh sách **tăng dần**?
- [ ] Không DFS đệ quy sâu ~1000?
- [ ] Đỉnh không có cạnh vào ⇒ `[]`?
- [ ] Chữ ký đúng: `def getAncestors(self, n: int, edges: List[List[int]]) -> List[List[int]]`?